In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score

In [3]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/pope_base_des_05_11_2025.csv")

In [3]:
preds = []
for i in data["answer"]:
    pred = i[:10].lower()
    if "yes" in pred:
        preds.append("yes")
    elif "no" in pred:
        preds.append("no")
    else:
        preds.append("unknown")
        
data["prediciton"] = preds
pd.Series(preds).value_counts()

yes    4817
no     4093
Name: count, dtype: int64

In [4]:
data.head(2)

,question,gt_answer,question_id,image_id,image_path,data_type,answer,prediciton
0,Is there a snowboard in the image?,yes,3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"Yes, there is a snowboard in the image, and th...",yes
1,Is there a backpack in the image?,no,14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"No, there is no backpack in the image. The per...",no


In [5]:
for name, group in data.groupby("data_type"):
    print(f"Type: {name}")
    # f1_scores = f1_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    acc_scores = accuracy_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    # print(f"F1 Score: {f1_scores}")
    print(f"Accuracy Score: {acc_scores}")

Type: adversarial
Accuracy Score: 0.7953333333333333
Type: popular
Accuracy Score: 0.8586666666666667
Type: random
Accuracy Score: 0.8917525773195877


In [6]:
accuracy_score(data["gt_answer"], preds)

0.8481481481481481

In [7]:
# tatget-word based evaluation

# tatget-word based evaluation

In [75]:
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/combined_stage_label_with_evidence_and_mlp_06_11_2025.pkl")

In [76]:
result_df["target_word"] = result_df["question"].apply(lambda x: x.replace("Is there a", "").replace("in the image?","").strip())

In [77]:
all_preds = []
for ans in result_df["answer"]:
    pred = ans[:10].lower()
    if "yes" in pred:
        all_preds.append("yes")
    elif "no" in pred:
        all_preds.append("no")
    else:
        all_preds.append("unknown")

In [78]:
result_df["pred_label"] = all_preds

In [79]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word,pred_label
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237482, 0....",snowboard,yes
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack,no


In [80]:
# only decision token based

In [92]:
all_new_labels = []
for inx, row in result_df.iterrows():
    old_label = row["pred_label"].lower()
    pred = row["labels_with_evidence"][0]["label"]
    if pred >= 0.5:
        new_label = old_label
    else:
        if old_label == "yes":
            new_label = "no"
        elif old_label == "no":
            new_label = "yes"
        else:
            new_label = "unknown"
    all_new_labels.append(new_label)
    

In [93]:
pd.Series(all_new_labels).value_counts()

no     4647
yes    4263
Name: count, dtype: int64

In [94]:
print(classification_report(result_df["gt_answer"], all_new_labels))

              precision    recall  f1-score   support

          no       0.85      0.90      0.87      4410
         yes       0.89      0.85      0.87      4500

    accuracy                           0.87      8910
   macro avg       0.87      0.87      0.87      8910
weighted avg       0.87      0.87      0.87      8910



In [95]:
result_df["corrected_labels"] = all_new_labels

In [96]:
for name, group in result_df.groupby("data_type"):
    print(f"Type: {name}")
    # f1_scores = f1_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    acc_scores = f1_score(group["gt_answer"].tolist(), group["corrected_labels"].tolist(), pos_label="yes")
    # print(f"F1 Score: {f1_scores}")
    print(f"f1 Score: {acc_scores}")

Type: adversarial
f1 Score: 0.8349769888231426
Type: popular
f1 Score: 0.8740536820371645
Type: random
f1 Score: 0.9015985790408526


# both decision token and evidence based

In [130]:
import torch

faild = []
corrrected_label = []
for inx, row in result_df.iterrows():
    old_label = row["pred_label"]
    try:
        prob = row["labels_with_evidence"][0]["label"]
        img_evi = (torch.tensor(row["labels_with_evidence"][0]["evidence"]) >= 0.4).int().sum().item()
        if prob >= 0.65:
            new_label = old_label
        elif img_evi >= 1 and prob < 0.65:
            new_label = old_label
        else:
            if old_label == "yes":
                new_label = "no"
            elif old_label == "no":
                new_label = "yes"
            else:
                new_label = "unknown"
        corrrected_label.append(new_label)

    except Exception as e:
        print(e)
        faild.append(inx)
        corrrected_label.append(old_label)

In [131]:
print(classification_report(result_df["gt_answer"], corrrected_label))

              precision    recall  f1-score   support

          no       0.84      0.91      0.87      4410
         yes       0.90      0.83      0.86      4500

    accuracy                           0.87      8910
   macro avg       0.87      0.87      0.87      8910
weighted avg       0.87      0.87      0.87      8910

